# Clase 3: pandas y Carga de Datos — Campo Volve

**Diplomado en Data Science Aplicada con Python para la Toma de Decisiones**  
SLB Ecuador · UDLA · 2026  

Instructor: **Carlos Enrique Mosquera Trujillo**  
Repo: [github.com/cmosquerat/slb-diplomado](https://github.com/cmosquerat/slb-diplomado)

---
## La ruta de hoy

1. Qué son los **datos** y los **datasets**.
2. Qué son las **librerías** y cómo se instalan (**pip**) e importan (**import**).
3. **pandas**: tablas de datos en Python (y un paréntesis: los **diccionarios**).
4. **Cargar archivos** reales en Colab (URL, subir, Drive) — un CSV del campo **Volve**.
5. Los **métodos básicos** de pandas: mirar, filtrar, resumir y **graficar**.
6. **Well logs**: el formato **LAS** y la librería `lasio`.
7. **Limpieza breve**: los datos faltantes (`NaN`).

> 💡 Ejecuta cada celda con `Shift + Enter`. Las celdas 🧩 de práctica están **en blanco**: te toca escribirlas.

---
# 1 · Los datos

**Un dato** es un hecho registrado: *"el 3 de junio de 2008 el pozo F-12 produjo 3 520 barriles"*. 
**Un dataset** es una colección organizada de miles de esos hechos, con una estructura que permite buscar, comparar y calcular.

### Datos tabulares
La estructura más común es la **tabla** (como una hoja de Excel):

- Cada **fila** es un **registro**: un día de un pozo, una medición.
- Cada **columna** es una **variable**: fecha, producción, presión.
- La **celda** es el cruce: un valor concreto.

### ¿Dónde viven? Los archivos
El formato más universal es el **CSV** (*Comma-Separated Values*): **texto plano** donde la primera línea trae los nombres de columna y cada línea siguiente es una fila:
```
fecha,pozo,oil
2008-06-01,15/9-F-12,3520
2008-06-01,15/9-F-14,2110
```
Existen otros formatos (Excel, JSON, LAS…). Hoy empezamos por el más simple y al final veremos uno **especializado de la industria petrolera**.

---
# 2 · Librerías: las cajas de herramientas de Python

Una **librería** es código que otra gente ya escribió, probó y regala, listo para que lo usemos — como una caja de herramientas: no fabricamos el martillo, lo tomamos y usamos.

| Librería | Para qué la usaremos hoy |
|----------|--------------------------|
| `pandas` | tablas de datos |
| `lasio`  | leer well logs (archivos LAS) |

Antes de usar una librería hay que hacer **dos cosas**:

### 1) Instalarla — `pip`, el gestor de librerías
`pip` es como la **tienda de aplicaciones** de Python: descarga e instala librerías desde el repositorio oficial (PyPI).
```
!pip install lasio
```
- El `!` le dice a Colab: *esto es un comando del sistema, no código Python*.
- Se instala **una vez por sesión** de Colab.
- **Colab ya trae preinstaladas** las más famosas (`pandas` incluida) — solo instalaremos las especializadas, como `lasio` (lo haremos más abajo, cuando la necesitemos).

### 2) Importarla — `import`
Instalar la deja en el computador; **importar** la trae a nuestro código:

In [ ]:
import pandas as pd

print("pandas importado, version:", pd.__version__)

- `as pd` crea un **alias**: un apodo corto para no escribir `pandas.` todo el tiempo.
- `pd` es la convención **universal**: úsenla siempre.
- Se importa **una vez por cuaderno**, normalmente en la primera celda.

> Regla de tres pasos: **instalar** (`pip`, si hace falta) → **importar** (`import`) → **usar**.

---
# 3 · pandas: tablas de datos en Python

El objeto central de pandas es el **DataFrame**: una tabla con filas y columnas *con nombre*. Una sola columna se llama **Series**.
```
DataFrame -> toda la tabla
Series    -> una columna
```

### Paréntesis: los diccionarios
Para crear tablas necesitamos un tipo nuevo de Python: el **diccionario**. Asocia una **etiqueta (clave)** a un **valor** — como un diccionario de verdad: buscamos por la palabra, no por posición.

- Se crea con **llaves** `{ }` y pares `"clave": valor`.
- La lista accede por número (`lista[0]`); el diccionario, por nombre (`pozo["bopd"]`).

In [ ]:
pozo = {
    "nombre": "Sacha-042",
    "bopd": 1250,
}

print(pozo["nombre"])
print(pozo["bopd"])

### Nuestro primer DataFrame

Las **listas paralelas** de la Clase 2 + un diccionario = una tabla. Cada **clave** es el nombre de una columna; su lista, los valores.

In [ ]:
datos = {
    "pozo": ["Sacha-042", "Sacha-108", "Auca-63"],
    "bopd": [1250, 780, 340],
    "corte": [0.32, 0.55, 0.78],
}

df = pd.DataFrame(datos)
df

El 0, 1, 2 de la izquierda es el **índice**: el número de cada fila. pandas lo agrega solo.

---
## 🧩 Práctica 1: Mi primera tabla

Creen su propio DataFrame con datos de 3 equipos de superficie:

1. Armen un **diccionario** con tres columnas: `equipo` (bomba, separador, compresor), `horas_uso` (números) y `estado` (texto).
2. Conviértanlo en DataFrame con `pd.DataFrame(...)`.
3. Muéstrenlo.

> Recuerden: las tres listas deben tener el **mismo largo**.

In [ ]:
# Escribe tu solucion aqui


---
# 4 · Cargar archivos en Colab

## El dataset de hoy: producción del campo Volve

**Volve** fue un campo de petróleo de **Equinor** en el Mar del Norte (Noruega). Produjo de **2008 a 2016** y en 2018 Equinor liberó *todos* sus datos al público. 
Preparamos un **CSV sencillo** a partir del Excel original: **15 634 filas** con la producción diaria de **7 pozos**.

| Columna | Qué es |
|---------|--------|
| `fecha` | día de la medición |
| `pozo`  | nombre del pozo |
| `horas` | horas en operación ese día |
| `oil`   | petróleo producido (Sm³/día) |
| `gas`   | gas producido (Sm³/día) |
| `agua`  | agua producida (Sm³/día) |

*Sm³ = metros cúbicos estándar, la unidad de volumen de la industria en Noruega.*

## Método 1: leer desde una URL (lo que haremos hoy)
Si el archivo está en internet, pandas lo **descarga y lee en un solo paso**:

In [ ]:
url = "https://raw.githubusercontent.com/cmosquerat/slb-diplomado/main/datos/volve_produccion.csv"

prod = pd.read_csv(url)
print(prod.shape)

## Método 2: subir un archivo del computador

Para archivos que tenemos en el disco:

1. Click en el ícono de **carpeta** (barra izquierda de Colab).
2. Botón **Subir** y elegir el archivo.
3. Leerlo por su **nombre**: `pd.read_csv("volve_produccion.csv")`.

> ⚠️ El archivo subido **se borra** cuando se cierra la sesión de Colab.

## Método 3: montar Google Drive

Para trabajar siempre con los mismos archivos, sin subirlos cada vez:
```python
from google.colab import drive
drive.mount("/content/drive")

prod = pd.read_csv("/content/drive/MyDrive/datos.csv")
```
Colab pedirá permiso con su cuenta de Google; sus archivos aparecen bajo `/content/drive/MyDrive/` y **persisten** entre sesiones.

> 💡 Los tres métodos terminan igual: `pd.read_csv(ruta)` → un DataFrame listo para trabajar.

## Primer vistazo
Regla de oro: apenas se carga un dataset, **mirarlo**.

In [ ]:
prod.head()   # primeras 5 filas

¿Qué es ese `NaN`? Una celda **vacía**: en 2007 el campo aún no producía. Al final de la clase aprenderemos a **limpiarlos**.

---
## 🧩 Práctica 2: Cargar el dataset

1. Ya leíste el CSV arriba — ahora muestra las **últimas** 5 filas con `.tail()`. ¿De qué año son?
2. ¿Cuántas filas y columnas tiene? (`.shape`)
3. Muestra las **primeras 10** filas (`.head(10)`).

In [ ]:
# Escribe tu solucion aqui


---
# 5 · Los métodos básicos de pandas

Un **método** es una acción del DataFrame: se llama con punto, `prod.metodo()`.

## Mirar la tabla: `head`, `tail`, `shape`, `info`

- `head(n)` / `tail(n)`: una ojeada al **inicio** y al **final**.
- `shape` va **sin paréntesis**: es un dato, no una acción.
- `info()`: el **tipo** de cada columna y cuántos valores llenos tiene.

In [ ]:
prod.info()

Fíjense: `oil` tiene solo 9 161 valores llenos de 15 634 filas — el resto son `NaN`.

## `.describe()`: estadísticas de una columna

- `count`: cuántos valores **no vacíos** hay.
- `mean`: el promedio. `50%`: la mediana.
- `min` y `max`: los extremos.

In [ ]:
prod["oil"].describe()

El mejor día de un pozo de Volve: **5 901 Sm³** de petróleo.

## Seleccionar columnas

Igual que en el diccionario: se pide la columna **por su nombre**.

- Corchete simple `[ ]`: **una** columna (una Series).
- Corchete doble `[[ ]]`: una **lista** de columnas.
- `.unique()`: los valores **distintos**.

In [ ]:
print(prod["pozo"].unique())   # los 7 pozos del campo

prod[["pozo", "oil"]].head(3)  # dos columnas

## Filtrar filas: una condición adentro

El patrón filtro de la Clase 2 (`for` + `if` + `append`), ahora en **una línea**: adentro va una condición y pandas devuelve **solo las filas** donde es `True`.

In [ ]:
dias_altos = prod[prod["oil"] > 4000]
print(len(dias_altos))

f12 = prod[prod["pozo"] == "15/9-F-12"]
print(len(f12))

626 días con más de 4 000 Sm³; 3 056 días registrados del pozo F-12.

## Crear una columna calculada

Operar columnas crea columnas nuevas. La operación se aplica a **todas las filas a la vez** (sin `for`: esto se llama *vectorización*).

In [ ]:
prod["liquido"] = prod["oil"] + prod["agua"]
prod[["oil", "agua", "liquido"]].tail(3)

> ⚠️ `NaN + NaN = NaN`: los huecos se **propagan**. Otra razón para limpiarlos (ya viene).

## `groupby`: agrupar y resumir

*"Divide las filas por pozo y suma el oil de cada grupo"* — el acumulador por pozo de la Clase 2, en dos líneas. También existen `.mean()`, `.max()`, `.count()`.

In [ ]:
grupos = prod.groupby("pozo")
totales = grupos["oil"].sum()
print(totales)

Dato real: **F-4 produjo 0**. Es un pozo **inyector** (mete agua al yacimiento para empujar el petróleo), no un productor.

## Graficar una columna: la idea de `.plot()`

Todo DataFrame sabe graficarse solo. Empecemos con la tabla pequeña de 3 pozos:

- `x=` y `y=`: qué columna va en cada eje.
- `kind=`: el tipo de gráfico (`"bar"` barras, `"barh"` horizontales, `"line"` línea).

In [ ]:
df.plot(x="pozo", y="bopd", kind="bar")

## Ahora con datos de verdad: la curva de declive

Detalle previo: `to_datetime` convierte el **texto** del CSV en **fechas de verdad**, para que el eje X ordene bien.

In [ ]:
prod["fecha"] = pd.to_datetime(prod["fecha"])

f12 = prod[prod["pozo"] == "15/9-F-12"]
f12.plot(x="fecha", y="oil")

La **curva de declive**: alta al inicio, cayendo con los años. Así viven y mueren los pozos.

## Graficar un `groupby`: barras

In [ ]:
totales.plot(kind="barh")

Dos pozos (F-12 y F-14) produjeron **el 85%** de todo el campo.

---
## 🧩 Práctica 3: Operar y graficar

Con la producción (`prod`) ya cargada:

1. ¿Cuál es la producción **promedio** de oil del campo? (`.mean()`)
2. ¿Y el **describe** completo de la columna `gas`?
3. Filtra los días del pozo `15/9-F-14` (`==`).
4. Grafica la producción de F-14 contra la fecha (`.plot(x=..., y=...)`).
5. Con `groupby`, calcula el **agua** total por pozo y grafícala en barras.

In [ ]:
# Escribe tu solucion aqui


---
# 6 · Well logs y el formato LAS

## ¿Qué es un well log (registro de pozo)?

Al perforar, se baja una herramienta que **mide las propiedades de la roca** a lo largo del pozo, cada pocos centímetros. El eje no es el tiempo: es la **profundidad** (metros). Cada propiedad medida es una **curva**.

Las curvas responden: *¿qué tipo de roca hay a cada profundidad? ¿tiene poros donde acumular petróleo? esos poros, ¿tienen petróleo o agua?* — es el dato con el que se decide dónde completar y producir un pozo.

## El formato LAS: el estándar de la industria

**LAS** = *Log ASCII Standard*. Desde 1990, todo software petrolero lo lee y escribe. Es **texto**, con una **cabecera** (pozo, campo, operador, unidades) y una **tabla de datos**:
```
~Well Information
STRT.M   102.1568 : Top
STOP.M  4636.5140 : Bottom
STEP.M     .15240 : Increment
NULL.    -999.250 : Null Value
WELL.     15/9-19 : NAME
COMP.     STATOIL : OPERATOR
~ASCII
 102.15  -999.25  -999.25 ...
```

### LAS vs CSV

| | CSV (producción) | LAS (well log) |
|---|---|---|
| Cada fila es… | un **día** de un pozo | una **profundidad** del pozo |
| El "eje" es… | el tiempo (fechas) | la profundidad (metros) |
| Espaciado | un registro por día | un registro cada **15.24 cm** |
| Metadatos | ninguno | **cabecera completa** con unidades |
| Dato faltante | celda vacía | código `-999.25` |
| Quién lo usa | cualquier industria | **estándar petrolero** |

La buena noticia: una vez cargados en Python, **los dos se vuelven DataFrames** y se manejan igual.

### Las dimensiones de nuestro LAS (pozo 15/9-19)

| Dimensión | Valor real |
|-----------|-----------|
| Profundidad inicial | 102.16 m |
| Profundidad final | 4 636.51 m |
| Paso de medición | 0.1524 m (= 6 pulgadas) |
| **Filas (profundidades)** | **29 754** |
| **Columnas (curvas)** | **7** |

En el CSV el índice era 0, 1, 2… Aquí el índice **es la profundidad**. 29 754 × 7 ≈ **208 mil mediciones** en un solo pozo.

### Las 7 curvas: qué nos dice cada columna

| Curva | Unidad | Qué mide | Qué nos dice |
|-------|--------|----------|--------------|
| `GR`   | GAPI | radioactividad natural | arcilla (alta) vs arena (baja) |
| `DEN`  | g/cm³ | densidad de la roca | porosidad |
| `NEU`  | % | respuesta al neutrón | porosidad (hidrógeno) |
| `AC`   | µs/ft | velocidad del sonido | porosidad, dureza |
| `RDEP` | ohm·m | resistividad profunda | **¿petróleo o agua?** |
| `RMED` | ohm·m | resistividad media | compara con RDEP |
| `CALI` | pulgadas | diámetro del hueco | calidad de la medición |

> 💡 La clave de la resistividad: el agua salada **conduce** electricidad (resistividad baja); el petróleo **no conduce** (alta). Un RDEP alto en roca porosa = zona de interés.

## `lasio`: la librería para leer LAS

Los tres pasos de toda librería: **instalar → importar → usar**. `lasio` **no viene** en Colab: por eso el `pip install`.

In [ ]:
!pip install lasio -q

In [ ]:
import lasio

url_las = "https://raw.githubusercontent.com/cmosquerat/slb-diplomado/main/datos/volve_15-9-19.LAS"

las = lasio.read(url_las)
log = las.df()   # a DataFrame

print(log.shape)
print(list(log.columns))

`.df()` convierte el LAS en un **DataFrame**: cabecera afuera, curvas adentro. Los `-999.25` se vuelven `NaN` automáticamente.

## Graficar una curva del log

En el log, el índice es la profundidad: `.plot()` la pone automáticamente en el eje X. Los picos de GR = zonas con más **arcilla**; los valles = **arenas** (posibles reservorios).

In [ ]:
log["GR"].plot()

> Los petrofísicos lo dibujan en **vertical**; lo aprenderemos más adelante en el diplomado.

---
## 🧩 Práctica 4: Abrir el well log

1. ¿Cuántas filas tiene el log? ¿Qué curvas trae? (`.shape`, `.columns`)
2. Ver `.head()`: ¿por qué hay tantos `NaN` al inicio? *(pista: las herramientas no miden desde la superficie; cada curva cubre solo un tramo)*
3. Saca el `.describe()` de la curva `GR`: ¿mínimo, máximo, media?
4. Grafica la curva `DEN` con `.plot()`. ¿En qué tramo del pozo hay datos?

In [ ]:
# Escribe tu solucion aqui


---
# 7 · Limpieza de datos (breve pero esencial)

## Contar los faltantes primero

Ya vimos los `NaN`. El primer paso siempre es saber **cuántos hay y dónde**: `.isna()` marca cada celda vacía y `.sum()` las cuenta por columna.

In [ ]:
prod.isna().sum()

**6 473** días sin producción registrada (de 15 634): pozos aún no perforados o cerrados. Es **normal** en datos reales — lo importante es decidir qué hacer.

## Dos remedios: borrar o rellenar

- `dropna()` **elimina** las filas con huecos (rápido, pero bota información).
- `fillna(valor)` los **completa**. Pozo cerrado = 0 producción: aquí rellenar con 0 **tiene sentido físico**. En otros datos podría ser el promedio, o interpolar entre vecinos (`.interpolate()`, útil en logs).

> No hay receta única: la decisión depende de **qué significa** el hueco.

In [ ]:
limpio = prod.dropna()
print(prod.shape)     # antes
print(limpio.shape)   # despues

In [ ]:
prod["oil"] = prod["oil"].fillna(0)
prod["gas"] = prod["gas"].fillna(0)
prod["agua"] = prod["agua"].fillna(0)

prod.isna().sum()

---
## 🧩 Práctica integradora: Reporte Volve

Todo lo de hoy, junto: el mismo reporte de la Clase 2, ahora con 15 634 filas reales. Parte de cero (vuelve a cargar el CSV en una variable nueva):

1. **Cargar** el CSV de producción desde la URL.
2. **Contar** los faltantes por columna con `.isna().sum()`.
3. **Rellenar** con 0 los `NaN` de `oil`, `gas` y `agua`.
4. **Convertir** `fecha` con `pd.to_datetime`.
5. **Resumir**: producción total de oil por pozo (`groupby`). ¿Cuál produjo más?
6. **Graficar**: la producción del mejor pozo contra la fecha, y los totales en barras.

> En la Clase 2, el reporte de 5 pozos nos tomó ~25 líneas. Hoy, con 15 634 filas, son ~8.

In [ ]:
# Escribe tu reporte aqui


---
## Cierre

**Conceptos:** dato, dataset y datos tabulares · qué es una librería; `pip` e `import` · diccionarios `{clave: valor}` · DataFrame y Series · qué es un well log y el formato **LAS** · qué mide cada curva (GR, DEN, RDEP…) · qué es un `NaN`.

**Habilidades:** cargar CSV por URL, subida o Drive · leer un LAS con `lasio` · `head`, `tail`, `shape`, `info`, `describe`, `unique` · filtrar filas y crear columnas · resumir con `groupby` · graficar con `.plot()` (línea y barras) · limpiar con `dropna` / `fillna`.

**Habilidad clave:** tomar un archivo real, cargarlo, limpiarlo y sacar conclusiones.

---
Carlos Enrique Mosquera Trujillo · cmosquerat@unal.edu.co · SLB Ecuador · UDLA · 2026

*Datos: campo Volve, Equinor (dataset abierto, 2018).*